## Real-world-grade problem:
### 🏥 Patient Medical Records System
Build a Pydantic model system for a hospital's patient intake and record management.


Requirements
```
1. Address model

street, city, state — all required strings
pincode — exactly 6 digits (string, validated)
country — default "India"

2. EmergencyContact model

name — required string
relation — must be one of: "spouse", "parent", "sibling", "friend"
phone — 10-digit string, validated

3. Medication model

name — required string
dosage_mg — positive float
frequency_per_day — int, between 1 and 4

4. Patient model

patient_id — auto-generated UUID, not supplied by user
name — min 2 chars, auto title-cased on input
age — int, between 0 and 120
blood_group — must be one of: A+, A-, B+, B-, O+, O-, AB+, AB-
address — nested Address model
emergency_contact — nested EmergencyContact
medications — list of Medication, can be empty, default empty list
admitted_on — datetime, default to current time
discharge_on — datetime | None, default None, must be after admitted_on if provided
is_critical — bool, default False
notes — str | None, default None

```

Tasks

1. Create the models with all validations
2. Instantiate a valid Patient and dump to a JSON file
3. Try instantiating with 3 different invalid inputs and catch the ValidationError — print a clean error message for each
4. Write a function discharge_patient(patient, discharge_time) that returns a new patient object with discharge_on set — without mutating the original


Constraints

- Use field validators wherever needed (@field_validator)
- Use model validator for the admitted_on / discharge_on cross-field check (@model_validator)
- Use Literal or Annotated where appropriate
- No raw dict manipulation — everything through Pydantic


Bonus

- Add a summary() method on Patient that prints a clean human-readable report
- Reject discharge_on that is in the past relative to admitted_on with a custom error message



#### 0. Necessary Imports as we go

In [1]:
from pydantic import BaseModel, Field, field_validator, model_validator, AwareDatetime, computed_field, ConfigDict, ValidationError
from typing import Annotated, Literal, Any

#### 1. Create the models with all validations

In [2]:
class Address(BaseModel):
    
    # intenstionally doing this to test out int pincode input
    model_config = ConfigDict(coerce_numbers_to_str=True) # not working as str->int okay but not int->str
    # this variable assignment is necesary

    street: str 
    city: str
    state: str
    # exactly 6 digits (string, validated) so here i can do max & min length=6 or should i do custom field_validator !?
    pincode: Annotated[str, Field(..., max_length=6, min_length=6)]  
    country: str = "India"


In [3]:
class EmergencyContact(BaseModel):
    name: str
    relation: Literal["spouse", "parent", "sibling", "friend"]
    phone: str # 10-digit string, validated - here ill be doing field_validator

    @field_validator('phone', mode="after")   # NameError: name 'phone' is not defined !!WHYYY?? - bcoz missed ""
    @classmethod
    def phone_verifier(cls, value):
        if len(value)!=10:
            raise ValueError("Phone number must be 10 digits long")
        return value

In [4]:
class Medication(BaseModel):
    name : str
    dosage_mg: Annotated[float, Field(ge=0)]
    frequency_per_day: Annotated[int, Field(ge=1, le=4)]

In [5]:
from datetime import datetime, timezone # we need the class below
import uuid

class Patient(BaseModel):
    # patient_id — auto-generated UUID, not supplied by user
    patient_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    name: str  # — min 2 chars, auto title-cased on input - will do field_validation as little complex logic
    age: Annotated[int, Field(ge=0, le=120)] # — int, between 0 and 120
    blood_group: Literal["A+", 'A-', 'B+', 'B-', 'O+', 'O-', 'AB+', 'AB-']
    address: Address # nested Address model
    emergency_contact: EmergencyContact
    # medications: Annotated[list[Medication]|None, Field(default_factory=list[str])] # list of Medication, can be empty, default empty list ??? im doing correctly? fyi empty list, even optional working
    medications: list[Medication] = Field(default_factory=list)  # CORRECTED, as requirement says "can be empty", not "can be None".
    # admitted_on:   # — datetime, default to current time - i think need to be a computed_field lets see with datetime.now default factory !
    admitted_on: Annotated[ datetime,  Field(default_factory=datetime.now) ] # the func should not be called yet
    # discharge_on:  # — datetime | None, default None, must be after admitted_on if provided - so Ill create model_validator
    discharge_on: datetime|None = None # is AwareDatetime correct to use here?
    is_critical : bool = False
    notes: str | None = None

    # @computed_field
    # @property                       # IMP: generates new UUID each time accessed, so use default_factory instead
    # def patient_id(self) -> str:    # Was Error - Computed field is missing return type annotation, not sure about return type, so using Any
    #     return uuid.uuid4()
    
    @field_validator('name', mode="before")
    @classmethod
    def name_beautifier_and_checker(cls, nm) -> str:   # auto title-cased on input
        if len(nm)<2:
            raise ValueError("Name length must be 2 characters atleast")
        return nm.title()

    @model_validator(mode='after')
    def discharge_on_validation(self):
        if self.discharge_on !=None and self.discharge_on < self.admitted_on:
            raise ValueError("Patient Rejected! must be after admitted_on if provided")
        return self

    def summary(self): # as per bonus suggestion - human readable funct
        all_info = self.model_dump()
        from pprint import pprint
        pprint(all_info)

#### 2. Instantiate a valid Patient and dump to a JSON file

In [6]:
address1_inputs = {
    "street": "chaibasa road",
    "city": "patna",
    "state": "bihar",
    "pincode": '792132', # intensionally giving int instead of str
    # country: str = "India"
}

address1 = Address(**address1_inputs)   # a good way to pass inputs
address1

Address(street='chaibasa road', city='patna', state='bihar', pincode='792132', country='India')

In [7]:
emecont1 = EmergencyContact(name="Yamasus Rakahbmuk", relation="friend", phone="9933538899")
emecont1

EmergencyContact(name='Yamasus Rakahbmuk', relation='friend', phone='9933538899')

In [8]:
med1 = Medication(name="dolo 650", dosage_mg=650, frequency_per_day=3)
med2 = Medication(name="Domperidone", dosage_mg=10, frequency_per_day=2)

med1, med2

(Medication(name='dolo 650', dosage_mg=650.0, frequency_per_day=3),
 Medication(name='Domperidone', dosage_mg=10.0, frequency_per_day=2))

In [9]:
patient1_inputs = {
    # patient_id — auto-generated 
    "name": "yoKita GOYAL",
    "age": 45,
    "blood_group": 'O+',
    "address": address1, # the earlier created Address model obj
    "emergency_contact": emecont1, # same 
    "medications": [med2, med1], # list of Medication, can be empty
    # admitted_on:   # — datetime, default to current time
    # "discharge_on": None,  # — datetime | None, default None, must be after admitted_on if provided
    "is_critical" : False,
    "notes": "She needs to do blood test and USG after 4 days if problem persists",

}

patient1 = Patient(**patient1_inputs)
patient1

Patient(patient_id='1c4d2291-1b88-4a91-913b-55a0b7051090', name='Yokita Goyal', age=45, blood_group='O+', address=Address(street='chaibasa road', city='patna', state='bihar', pincode='792132', country='India'), emergency_contact=EmergencyContact(name='Yamasus Rakahbmuk', relation='friend', phone='9933538899'), medications=[Medication(name='Domperidone', dosage_mg=10.0, frequency_per_day=2), Medication(name='dolo 650', dosage_mg=650.0, frequency_per_day=3)], admitted_on=datetime.datetime(2026, 6, 16, 23, 28, 13, 793180), discharge_on=None, is_critical=False, notes='She needs to do blood test and USG after 4 days if problem persists')

In [10]:
with open("patient1_dump.json","w+") as fp:
    fp.write(patient1.model_dump_json(indent=4))

#### 3. Try instantiating with 3 different invalid inputs and catch the ValidationError — print a clean error message for each

In [11]:
patient2_inputs = {
    "name": "yoKita GOYAL",
    "age": 45,
    "blood_group": 'Z+', # NOT POSSIBLE
    "address": address1, 
    "emergency_contact": emecont1, 
    "medications": [med2, med1], 
    "is_critical" : False,
    "notes": "She needs to do blood test and USG after 4 days if problem persists",
}

try:
    patient2 = Patient(**patient2_inputs)
except ValidationError as e:
    for err in e.errors():
        print(f"Field: {err['loc']} | Error: {err['msg']}")

Field: ('blood_group',) | Error: Input should be 'A+', 'A-', 'B+', 'B-', 'O+', 'O-', 'AB+' or 'AB-'


In [12]:
patient3_inputs = {
    "name": "yoKita GOYAL",
    "age": 45,
    "blood_group": 'O+',
    "address": 'dsa',  # hardcoding val
    "emergency_contact": emecont1, 
    "medications": [med2, med1], 
    "is_critical" : False,
    "notes": "She needs to do blood test and USG after 4 days if problem persists",
}


try:
    patient3 = Patient(**patient3_inputs)
except ValidationError as e:
    for err in e.errors():
        print(f"Field: {err['loc']} | Error: {err['msg']}")

Field: ('address',) | Error: Input should be a valid dictionary or instance of Address


In [13]:
address2_inputs = {
    "street": "chaibasa road",
    "city": "patna",
    "state": "bihar",
    "pincode": '1234567', # giving 7 digit
    "country": "Maylesia"
}

try:
    address2 = Address(**address2_inputs)
except ValidationError as e:
    for err in e.errors():
        print(f"Field: {err['loc']} | Error: {err['msg']}")

Field: ('pincode',) | Error: String should have at most 6 characters


In [14]:
try:
    emecont2 = EmergencyContact(name="Yamasus Rakahbmuk", relation="friend", phone="99335388991")

except ValidationError as e:
    for err in e.errors():
        print(f"Field: {err['loc']} | Error: {err['msg']}")

Field: ('phone',) | Error: Value error, Phone number must be 10 digits long


#### 4. Write a function discharge_patient(patient, discharge_time) that returns a new patient object with discharge_on set — without mutating the original

In [15]:
def discharge_patient(patient: Patient, discharge_time: datetime) -> Patient:

    # The below violets the requirement
    # patient_new = patient.model_dump(exclude_none=True, exclude_defaults=True, exclude_unset=False)
    # patient_new["discharge_on"] = discharge_time
    # patient_new_pydantic_obj = Patient(**patient_new) # bcoz i need model validation as discharge_on>admitted_on
    # return patient_new_pydantic_obj

    return patient.model_copy(update={"discharge_on": discharge_time})

In [16]:
print(discharge_patient(patient=patient1, discharge_time=datetime.now())) # error - Input should have timezone info!

patient_id='1c4d2291-1b88-4a91-913b-55a0b7051090' name='Yokita Goyal' age=45 blood_group='O+' address=Address(street='chaibasa road', city='patna', state='bihar', pincode='792132', country='India') emergency_contact=EmergencyContact(name='Yamasus Rakahbmuk', relation='friend', phone='9933538899') medications=[Medication(name='Domperidone', dosage_mg=10.0, frequency_per_day=2), Medication(name='dolo 650', dosage_mg=650.0, frequency_per_day=3)] admitted_on=datetime.datetime(2026, 6, 16, 23, 28, 13, 793180) discharge_on=datetime.datetime(2026, 6, 16, 23, 28, 13, 920419) is_critical=False notes='She needs to do blood test and USG after 4 days if problem persists'


#### 5. [Bonus] Add a summary() method on Patient that prints a clean human-readable report

In [17]:
patient1.summary()

{'address': {'city': 'patna',
             'country': 'India',
             'pincode': '792132',
             'state': 'bihar',
             'street': 'chaibasa road'},
 'admitted_on': datetime.datetime(2026, 6, 16, 23, 28, 13, 793180),
 'age': 45,
 'blood_group': 'O+',
 'discharge_on': None,
 'emergency_contact': {'name': 'Yamasus Rakahbmuk',
                       'phone': '9933538899',
                       'relation': 'friend'},
 'is_critical': False,
 'medications': [{'dosage_mg': 10.0,
                  'frequency_per_day': 2,
                  'name': 'Domperidone'},
                 {'dosage_mg': 650.0,
                  'frequency_per_day': 3,
                  'name': 'dolo 650'}],
 'name': 'Yokita Goyal',
 'notes': 'She needs to do blood test and USG after 4 days if problem persists',
 'patient_id': '1c4d2291-1b88-4a91-913b-55a0b7051090'}


#### 6. [BONUS] Reject discharge_on that is in the past relative to admitted_on with a custom error message

In [18]:
patient6_inputs = {
    "name": "yoKita GOYAL",
    "age": 45,
    "blood_group": 'O+',
    "address": address1,
    "emergency_contact": emecont1,
    "medications": [med2, med1],
    "admitted_on": "2026-10-10",
    "discharge_on": "2026-11-11",   # NB
    "is_critical" : True,
    "notes": "She needed ventilation support",
}

patient6 = Patient(**patient6_inputs)
patient6

Patient(patient_id='5664329b-56f4-4146-9d8b-1325647b43e4', name='Yokita Goyal', age=45, blood_group='O+', address=Address(street='chaibasa road', city='patna', state='bihar', pincode='792132', country='India'), emergency_contact=EmergencyContact(name='Yamasus Rakahbmuk', relation='friend', phone='9933538899'), medications=[Medication(name='Domperidone', dosage_mg=10.0, frequency_per_day=2), Medication(name='dolo 650', dosage_mg=650.0, frequency_per_day=3)], admitted_on=datetime.datetime(2026, 10, 10, 0, 0), discharge_on=datetime.datetime(2026, 11, 11, 0, 0), is_critical=True, notes='She needed ventilation support')

In [19]:
patient5_inputs = {
    "name": "yoKita GOYAL",
    "age": 45,
    "blood_group": 'O+',
    "address": address1,
    "emergency_contact": emecont1,
    "medications": [med2, med1],
    "admitted_on": "2026-10-10",
    "discharge_on": "2026-09-09",   # NB, its not possible
    "is_critical" : True,
    "notes": "She needed ventilation support",
}

patient5 = Patient(**patient5_inputs)
patient5

ValidationError: 1 validation error for Patient
  Value error, Patient Rejected! must be after admitted_on if provided [type=value_error, input_value={'name': 'yoKita GOYAL', ...ed ventilation support'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error